human lung cancer

导入相关包

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)


In [2]:
repeat_times = 2
shard_inx = [0,1,2]
simple_path = [f'/home/cavin/jt/benchmark/data/crc/VisiumHD_P2_shard_{shard_inx}.h5ad' for shard_inx in shard_inx]
cell_emb_col = 'X_scFoundation'
save_path = [f"/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_no_annotations/scfoundation_human_CRC_shard_{shard}_emb.parquet" for shard in shard_inx]


In [3]:
adata = [sc.read_h5ad(path) for path in simple_path]
var = adata[0].var
adata = sc.concat(adata, axis=0)
adata.var = var
del var
adata

AnnData object with n_obs × n_vars = 500470 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    obsm: 'spatial'

读取嵌入

In [4]:
loaded_emb_df = [pd.read_parquet(s) for s in save_path]
loaded_emb_df = pd.concat(loaded_emb_df)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
del loaded_emb_df
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 500470 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    obsm: 'spatial', 'X_scFoundation'

In [5]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
random_seed=2026 + repeat_times * 200
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 500470 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'neighbors'
    obsm: 'spatial', 'X_scFoundation'
    obsp: 'distances', 'connectivities'

In [6]:
def main(adata,random_seed,res_list,key_added):
    from tqdm import tqdm
    for used_res in tqdm(res_list):
        sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=f"{key_added}_{used_res}")


In [7]:
main(adata,random_seed,res_list,key_added)

  0%|                                              | 0/10 [00:00<?, ?it/s]

/tmp/ipykernel_2408232/2761875232.py:4: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=f"{key_added}_{used_res}")


 10%|███▋                                 | 1/10 [02:54<26:13, 174.89s/it]

 20%|███████▍                             | 2/10 [08:22<35:18, 264.77s/it]

 30%|███████████                          | 3/10 [14:23<35:59, 308.54s/it]

...

 70%|█████████████████████████▉           | 7/10 [43:31<20:19, 406.37s/it]

 80%|█████████████████████████████▌       | 8/10 [51:06<14:03, 421.90s/it]

 90%|███████████████████████████████▌   | 9/10 [1:00:39<07:49, 469.13s/it]

100%|██████████████████████████████████| 10/10 [1:09:24<00:00, 486.34s/it]

100%|██████████████████████████████████| 10/10 [1:09:24<00:00, 416.48s/it]

In [8]:
for used_res in res_list:
    save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/CRC/scfoundation_human_CRC_labels_repeat_{repeat_times}_resolution_{used_res}.csv"
    adata.obs[f"{key_added}_{used_res}"].to_csv(save_label_df_path)